<!-- Cell 0 -->
# Policy Intelligence POC — Run One Policy (CITI_ONLY)

Flow: **config → GuardRail (Llama 4) → run → archive report → offline pack** (no `*_FULL.zip`).
**CITI JupyterHub:** put the personal trading policy under `policies/`, set `POLICY_PATH` / `POLICY_ID` in Cell 4, then **Run All once**.

Outputs:
- Pipeline demo: `policy_runs/<POLICY_ID>/<RUN_ID>/demo/index.html`
- Offline pack (main + role views): `notebooks/offline_p2b/index.html` (section 3b)
- Offline zip for handoff: `notebooks/offline_p2b_OFFLINE_DEMO.zip` (section 3b or `pack_offline_demo.bat`)

**Offline views (one pack):**
- **Main** — `index.html` (BPMN workspace). Intelligence lens defaults to **All roles (main view)**.
- **Policy Follower / Compliance Officer / Policy Owner** — opens `role_intelligence.html` with workflows/BPMN from the same pack.

Pass F embeds `role_relevance` + `role_workflow_map` into `data.js` when Cell 8 produced a run.

This package uses **GuardRail only** via `import guardrail_last`. OpenAI is not included.


<!-- Cell 1 -->
## 0. Optional — confirm Hub packages

On CITI JupyterHub packages are preinstalled. Skip installs. Run Cell 2 only if a reviewer approved a local inventory check.


In [ ]:
# Cell 2
# CITI Hub: packages are preinstalled — do not pip-install unless approved.
# Inventory only (no openai):
import sys
print("python", sys.executable)
print("skip pip on Hub — requirements.txt is inventory for review")


<!-- Cell 3 -->
## 1. Bootstrap + config

1. Put your policy under `policies/` (pdf / docx / html).
2. Set `POLICY_ID` and `POLICY_PATH` below.
3. GuardRail: Cell 6 uses `from guardrail_last import GuardRail` (Hub class).


In [ ]:
# Cell 4
import os
import sys
from pathlib import Path

# --- Project root on sys.path (must come before policy_knowledge imports) ---
ROOT = Path.cwd()
if (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT / "src"))
elif (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT / "src"))

# Notebook helpers (offline p2b pack builder)
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))

from policy_knowledge.config import load_config, TransferPolicy


def load_dotenv(path: Path) -> None:
    """Load KEY=VALUE from .env into os.environ (does not override existing)."""
    if not path.is_file():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key, value = key.strip(), value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


load_dotenv(ROOT / ".env")
config = load_config(ROOT / "config" / "app.example.yaml")

POLICY_ID = "PTIP_BR_001"
POLICY_PATH = ROOT / "policies" / "PTIP_black_rock-21-2026.pdf"

# data.js always gets full parsed text (feeds demo/index.html).
# workstation_transfer_approved only gates creating a shareable DEMO ZIP.
transfer_policy = TransferPolicy(
    include_policy_excerpts_in_demo=True,
    workstation_transfer_approved=False,
)

print("ROOT", ROOT)
print("POLICY_ID", POLICY_ID)
print("POLICY_PATH", POLICY_PATH, "exists=", POLICY_PATH.exists())
print("SOEID set=", bool(os.environ.get("SOEID") or os.environ.get("POLICY_SOEID")))


<!-- Cell 5 -->
## 2. GuardRail (CITI Hub) — same pattern as Hub samples

```text
from guardrail_last import GuardRail
GuardRail.INITIALIZE(soeId)
generate = partial(GuardRail.CHAT_COMPLETIONS_CREATE, model=MODEL, user=soeId)
_, c = generate(messages=..., temperature=0, top_p=0.9)
```

`CHAT_COMPLETIONS_CREATE` / `PARSE` return `(request_id, completion)`. Falls back to mock only if GuardRail is unavailable.


In [ ]:
# Cell 6 — GuardRail call shape matches Hub samples (Python 3.9+)
from functools import partial
from typing import List, Optional

from policy_knowledge.intelligence.guardrail_client import (
    GuardRailUnavailable,
    create_guardrail_client,
    resolve_guardrail_backend,
)
from policy_knowledge.intelligence.llm_factory import llm_runtime_status
from policy_knowledge.intelligence.synthetic_mock import SyntheticMockLLM

# --- Hub sample knobs (override via env on shared kernels) ---
MODEL = os.environ.get("POLICY_LLM_MODEL", "Llama-4-Scout-17B-16E-Instruct")
soeId = os.environ.get("SOEID") or os.environ.get("POLICY_SOEID") or ""
GENERATION_TEMPERATURE = float(os.environ.get("POLICY_LLM_TEMPERATURE", "0"))

config.llm.provider = "llama4"
config.llm.model = MODEL
config.llm.narrative_model = MODEL
config.llm.temperature = GENERATION_TEMPERATURE


def build_messages(prompt, system_prompt=None):
    # type: (str, Optional[str]) -> List[dict]
    """Hub-style messages helper (user prompt + optional system)."""
    messages = [{"role": "user", "content": prompt}]
    if system_prompt is not None:
        messages.insert(0, {"role": "system", "content": system_prompt})
    return messages


generate = None
parse = None
get_generate = None

# --- Locate guardrail_last.py (Hub folder: guardrail_combined) ---
# Set GUARDRAIL_DIR in env to override. Folder must contain guardrail_last.py, Config.py, Service*.py, certs/
GUARDRAIL_DIR = None
for _cand in [
    os.environ.get("GUARDRAIL_DIR"),
    str(ROOT / "guardrail_combined"),
    str(ROOT / "notebooks" / "guardrail_combined"),
    str(ROOT.parent / "guardrail_combined"),
]:
    if _cand and (Path(_cand) / "guardrail_last.py").is_file():
        GUARDRAIL_DIR = Path(_cand)
        break
if GUARDRAIL_DIR is not None:
    sys.path.insert(0, str(GUARDRAIL_DIR))  # ahead of notebooks/ so the Hub file wins
    print("guardrail folder:", GUARDRAIL_DIR)
else:
    print("guardrail_combined not found; set GUARDRAIL_DIR to the folder with guardrail_last.py")

try:
    from guardrail_last import GuardRail  # Hub: class GuardRail in guardrail_last.py

    GuardRail = resolve_guardrail_backend(GuardRail)
    if not soeId:
        raise GuardRailUnavailable("Set SOEID or POLICY_SOEID before GuardRail.INITIALIZE")

    if GUARDRAIL_DIR is not None:
        os.chdir(GUARDRAIL_DIR)  # Hub module loads certs/ relative to cwd
    GuardRail.INITIALIZE(soeId)
    parse = partial(GuardRail.CHAT_COMPLETIONS_PARSE, model=MODEL, user=soeId)
    generate = partial(GuardRail.CHAT_COMPLETIONS_CREATE, model=MODEL, user=soeId)

    def get_generate(model):
        return partial(GuardRail.CHAT_COMPLETIONS_CREATE, model=model, user=soeId)

    # Pipeline adapter (unwraps Hub (request_id, completion) tuples)
    llm = create_guardrail_client(
        require_live=True,
        model=MODEL,
        narrative_model=MODEL,
        soeid=soeId,
        temperature=GENERATION_TEMPERATURE,
        backend=GuardRail,
    )

    # Optional smoke — same unpack as Hub samples: (_, completion)
    try:
        _req_id, _smoke = generate(
            messages=build_messages("Reply with OK only.", "You are a concise assistant."),
            temperature=GENERATION_TEMPERATURE,
            top_p=0.9,
        )
        _text = getattr(getattr(getattr(_smoke, "choices", [None])[0], "message", None), "content", None)
        print("GuardRail smoke OK", "request_id=", _req_id, "preview=", str(_text)[:80])
    except Exception as smoke_exc:  # noqa: BLE001
        print("GuardRail smoke skipped/failed:", smoke_exc)
except (ImportError, GuardRailUnavailable) as exc:
    print(exc)
    print("Falling back to mock — do NOT claim live Llama connectivity.")
    llm = SyntheticMockLLM()
    config.llm.provider = "mock"
except Exception as exc:  # noqa: BLE001
    print(exc)
    print("Falling back to mock — do NOT claim live Llama connectivity.")
    llm = SyntheticMockLLM()
    config.llm.provider = "mock"

status = llm_runtime_status(llm)
print(
    status["llm_status"],
    type(llm).__name__,
    "provider=",
    config.llm.provider,
    "model=",
    getattr(llm, "model", config.llm.model),
    "soeId=",
    bool(soeId),
)
if status["llm_status"] != "GUARDRAIL_CONNECTED":
    print("CITI: `from guardrail_last import GuardRail` + set SOEID / POLICY_LLM_MODEL")


<!-- Cell 7 -->
## 3. Run pipeline + archive

Writes `demo/data.js` (full text) + OKF/BPMN/matrices + `role_relevance.json`.
**No `*_FULL.zip`** — offline handoff is section **3b** → `offline_p2b_OFFLINE_DEMO.zip`. DEMO ZIP only if `workstation_transfer_approved=True`.


In [ ]:
# Cell 8
import json
print(transfer_policy)
from policy_knowledge.packaging.package_policy import archive_policy_run
from policy_knowledge.pipeline import run_policy_demo

# Ingest → extract → export OKF/BPMN/demo (data.js populated) → QA
result = run_policy_demo(
    POLICY_PATH,
    policy_id=POLICY_ID,
    config=config,
    llm=llm,
    transfer_policy=transfer_policy,
    overwrite=True,
)

demo_html = result.run_dir / "demo" / "index.html"
data_js = result.run_dir / "demo" / "data.js"
role_json = result.run_dir / "outputs" / "canonical" / "role_relevance.json"
print("run_dir", result.run_dir)
print("open HTML", demo_html)
print("data.js bytes", data_js.stat().st_size if data_js.exists() else 0)
print("role_relevance", role_json if role_json.exists() else "MISSING")
print("validation", json.dumps(result.validation, indent=2)[:2000])

# No *_FULL.zip (use section 3b offline_p2b zip). DEMO zip only if transfer approved.
archives = archive_policy_run(result.run_dir, transfer_policy=transfer_policy)
print("full_zip", archives.full_zip)  # always None
print("demo_zip", archives.demo_zip)
print("report", archives.report_path)

<!-- Cell 9 -->
## 3b. Offline pack (main + role views) + zip

Packages `policies2BPMN/outputs` into `notebooks/offline_p2b/`:
- **Main view** — `index.html`
- **Role views** — `role_intelligence.html` (Policy Follower / Compliance / Policy Owner)
- Embedded `data.js` with workflows, BPMN, Pass F lenses, and `role_workflow_map`

Also writes **`notebooks/offline_p2b_OFFLINE_DEMO.zip`** for offline handoff (or run `pack_offline_demo.bat` at repo root).

Open the printed `file://` URI. Does **not** modify `policies2BPMN/`.  
Requires `policies2BPMN/viewer/node_modules/bpmn-js` (`npm install` once in `policies2BPMN/viewer` if missing).


In [ ]:
# Cell 10
import importlib
import json
from pathlib import Path

import build_offline_p2b

importlib.reload(build_offline_p2b)  # pick up disk fixes without Kernel Restart
from build_offline_p2b import build_offline_p2b, zip_offline_pack

# Prefer the pipeline run from Cell 8 when available (Pass F lenses + role map).
run_dir = locals().get("result") and getattr(result, "run_dir", None)
build_kwargs = {}
if run_dir is not None:
    canonical = Path(run_dir) / "outputs" / "canonical"
    if (canonical / "role_relevance.json").is_file():
        build_kwargs["canonical_run_dir"] = Path(run_dir)

out = build_offline_p2b(ROOT, ROOT / "notebooks" / "offline_p2b", **build_kwargs)
zip_path = zip_offline_pack(out, ROOT / "notebooks" / "offline_p2b_OFFLINE_DEMO.zip")
html = out / "index.html"
manifest = json.loads((out / "OFFLINE_MANIFEST.json").read_text(encoding="utf-8"))
print("offline pack", out)
print("open main view", html.resolve().as_uri())
print("role views", manifest.get("role_views"))
print("workflows", len((out / "data.js").read_text(encoding="utf-8").split('"workflow_id"')) - 1)
print("role_relevance_embedded", manifest.get("role_relevance_embedded"))
print("role_workflow_map_embedded", manifest.get("role_workflow_map_embedded"))
print("offline_demo_zip", zip_path)
print("handoff: unzip offline_p2b_OFFLINE_DEMO.zip → open offline_p2b/index.html")


<!-- Cell 11 -->
## 4. Optional — run tests

In [ ]:
# Cell 12
import pytest

pytest.main(["-q", str(ROOT / "tests")])

In [ ]:
# Cell 13